# VoiceDiary AI — Live GPU Demo
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

> **Quick Start:** Click **Runtime → Run all** (`Ctrl+F9`) to start the live application.


In [ ]:
# Step 1: Install Dependencies & CUDA 12 Runtime Libraries
!pip install -q --no-cache-dir faster-whisper speechbrain gradio soundfile torchaudio scipy nvidia-cublas-cu12 nvidia-cudnn-cu12

# Configure system dynamic library paths so CTranslate2 immediately locates libcublas.so.12 & libcudnn
!python -c "import os, glob, nvidia.cublas.lib, nvidia.cudnn.lib; [os.symlink(f, f'/usr/lib/{os.path.basename(f)}') for d in [nvidia.cublas.lib, nvidia.cudnn.lib] for f in glob.glob(os.path.join(os.path.dirname(d.__file__), '*.so*')) if not os.path.exists(f'/usr/lib/{os.path.basename(f)}')]" 2>/dev/null || true
!ldconfig 2>/dev/null || true


In [ ]:
# Step 2: Load AI Models
import os, sys, time, json, re, urllib.request, base64, gc, ctypes, glob
import numpy as np
import soundfile as sf
import gradio as gr
import torch
import torchaudio
from scipy.signal import lfilter

# ── Dynamic CUDA 12 & cuBLAS / cuDNN Library Linker Fix ──
# Ensures CTranslate2 (faster-whisper) finds libcublas.so.12 and libcudnn across all Google Colab & Linux environments
def configure_cuda_libraries():
    search_paths = []
    try:
        import nvidia.cublas.lib
        search_paths.append(os.path.dirname(nvidia.cublas.lib.__file__))
    except Exception:
        pass
    try:
        import nvidia.cudnn.lib
        search_paths.append(os.path.dirname(nvidia.cudnn.lib.__file__))
    except Exception:
        pass
    try:
        torch_lib = os.path.join(os.path.dirname(torch.__file__), "lib")
        if os.path.isdir(torch_lib):
            search_paths.append(torch_lib)
    except Exception:
        pass
    for p in ["/usr/local/cuda/lib64", "/usr/local/cuda-12/lib64", "/usr/lib/x86_64-linux-gnu"]:
        if os.path.isdir(p):
            search_paths.append(p)

    search_paths = list(dict.fromkeys([p for p in search_paths if os.path.isdir(p)]))
    if search_paths:
        curr_ld = os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = ":".join(search_paths + ([curr_ld] if curr_ld else []))
        for d in search_paths:
            for lib in ["libcublasLt.so.12", "libcublas.so.12"]:
                p = os.path.join(d, lib)
                if os.path.isfile(p):
                    try:
                        ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL)
                    except Exception:
                        pass
            for f in sorted(os.listdir(d)):
                if ("cudnn" in f or "cublas" in f) and (".so" in f):
                    p = os.path.join(d, f)
                    if os.path.isfile(p):
                        try:
                            ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL)
                        except Exception:
                            pass
            # Symlink into /usr/lib if writable
            for f in glob.glob(os.path.join(d, "*.so*")):
                dest = f"/usr/lib/{os.path.basename(f)}"
                if not os.path.exists(dest):
                    try:
                        os.symlink(f, dest)
                    except Exception:
                        pass

configure_cuda_libraries()

from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

cuda_available = torch.cuda.is_available()
device_type = "cuda" if cuda_available else "cpu"
sb_device = "cuda:0" if cuda_available else "cpu"
compute_dtype = "float16" if cuda_available else "int8"
gpu_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
print(f"Device: {gpu_name} ({compute_dtype})")

_model_cache = {}
def get_model(name="large-v3-turbo"):
    global device_type, compute_dtype
    if name not in _model_cache:
        print(f"Loading Whisper model: {name}...")
        os.makedirs("/content/models/whisper", exist_ok=True)
        try:
            _model_cache[name] = WhisperModel(
                name,
                device=device_type,
                compute_type=compute_dtype,
                num_workers=1,
                cpu_threads=4,
                download_root="/content/models/whisper"
            )
        except RuntimeError as e:
            if ("libcublas" in str(e) or "cuda" in str(e).lower()) and device_type == "cuda":
                print(f"Warning: CUDA dynamic library issue encountered ({e}). Falling back to CPU int8...")
                _model_cache[name] = WhisperModel(
                    name,
                    device="cpu",
                    compute_type="int8",
                    num_workers=1,
                    cpu_threads=4,
                    download_root="/content/models/whisper"
                )
            else:
                raise e
    return _model_cache[name]

print("Loading Faster-Whisper Large-v3-Turbo...")
_warmup_whisper = get_model("large-v3-turbo")

print("Loading SpeechBrain ECAPA-TDNN...")
os.makedirs("/content/models/ecapa", exist_ok=True)
embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/models/ecapa",
    run_opts={"device": sb_device}
)

print("Loading Silero VAD...")
silero_vad_model, _ = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    onnx=False,
    trust_repo=True
)
silero_vad_model = silero_vad_model.to(device_type)
silero_vad_model.eval()

# GPU Pre-warm forward passes to eliminate first-chunk latency
print("Pre-warming GPU kernels...")
with torch.inference_mode():
    _dummy_vad = torch.zeros(1, 512).to(device_type)
    _ = silero_vad_model(_dummy_vad, 16000)

    _dummy_audio = torch.zeros(1, 16000).to(device_type)
    _ = embedder.encode_batch(_dummy_audio)

    try:
        _dummy_pcm = np.zeros(16000, dtype=np.float32)
        list(_warmup_whisper.transcribe(_dummy_pcm, language="en", beam_size=1, without_timestamps=True)[0])
    except Exception as e:
        print(f"Warning during Whisper GPU warm-up: {e}")
        if "libcublas" in str(e) and device_type == "cuda":
            print("Switching Whisper model to CPU fallback...")
            _model_cache["large-v3-turbo"] = WhisperModel(
                "large-v3-turbo",
                device="cpu",
                compute_type="int8",
                num_workers=1,
                cpu_threads=4,
                download_root="/content/models/whisper"
            )

print("Ready.")


In [ ]:
# Step 3: Audio Processing, Speaker Diarization & AI Lecture Summarizer

# 1. 80Hz High-Pass Filter & Audio Metrics
def butter_highpass_filter(data, cutoff=80.0, fs=16000.0):
    if len(data) < 32:
        return data
    rc = 1.0 / (2.0 * np.pi * cutoff)
    dt = 1.0 / fs
    alpha = rc / (rc + dt)
    y = np.empty_like(data)
    y[0] = data[0]
    for i in range(1, len(data)):
        y[i] = alpha * (y[i-1] + data[i] - data[i-1])
    return y.astype(np.float32)

def compute_rms(audio):
    if len(audio) == 0: return 0.0
    return float(np.sqrt(np.mean(audio ** 2)))

def l2_normalize(x: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(x)
    return x / (norm if norm > 0 else 1.0)

# Resampler Cache
_resampler_cache = {}
def resample_to_16k(audio_arr, orig_sr):
    if orig_sr == 16000:
        return audio_arr
    if orig_sr not in _resampler_cache:
        _resampler_cache[orig_sr] = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=16000)
    t_y = torch.from_numpy(audio_arr).float().unsqueeze(0)
    return _resampler_cache[orig_sr](t_y).squeeze().numpy()

# 2. Multi-Frame Voice Activity Detector with Adaptive Noise Floor Discrimination
def is_speech_in_chunk(
    chunk: np.ndarray,
    vad_model,
    dev_type="cuda",
    ambient_rms: float = 0.003,
    vad_threshold: float = 0.45
) -> tuple:
    if len(chunk) < 512:
        return False, ambient_rms, 0.0

    chunk_rms = compute_rms(chunk)
    if chunk_rms < max(0.004, ambient_rms * 1.35):
        updated_ambient = 0.92 * ambient_rms + 0.08 * chunk_rms
        return False, updated_ambient, 0.0

    num_frames = len(chunk) // 512
    if num_frames == 0:
        return False, ambient_rms, 0.0

    frames = chunk[:num_frames * 512].reshape(num_frames, 512)

    with torch.inference_mode():
        t_frames = torch.from_numpy(frames).float().to(dev_type)
        probs = []
        for i in range(num_frames):
            p = float(vad_model(t_frames[i:i+1], 16000).item())
            probs.append(p)

    if not probs:
        return False, ambient_rms, 0.0

    speech_frame_count = sum(1 for p in probs if p >= vad_threshold)
    speech_density = speech_frame_count / len(probs)
    max_prob = max(probs)
    mean_prob = float(np.mean(probs))

    is_speech = (speech_density >= 0.20 and max_prob >= (vad_threshold + 0.05)) or (mean_prob >= 0.52)

    if not is_speech:
        updated_ambient = 0.90 * ambient_rms + 0.10 * chunk_rms
    else:
        updated_ambient = ambient_rms

    return is_speech, updated_ambient, max_prob

# 3. Speaker Diarization Matcher (Multi-Exemplar Engine)
class SpeakerMatcher:
    def __init__(self, threshold=0.32):
        self.threshold = threshold
        self.speakers = {}
        self.centroids = {}
        self.matrices = {}
        self.last_speaker_id = None
        self.last_speaker_time = 0.0
        self.colors = [
            '#6366F1', '#10B981', '#F59E0B', '#EC4899', '#06B6D4',
            '#8B5CF6', '#F97316', '#14B8A6', '#3B82F6', '#E11D48'
        ]

    def reset(self):
        self.speakers.clear()
        self.centroids.clear()
        self.matrices.clear()
        self.last_speaker_id = None
        self.last_speaker_time = 0.0

    def _rebuild_cache(self):
        self.centroids.clear()
        self.matrices.clear()
        for sid, data in self.speakers.items():
            embs = data["embeddings"]
            if not embs: continue
            raw_mat = np.array(embs, dtype=np.float32)
            norms = np.linalg.norm(raw_mat, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            norm_mat = raw_mat / norms
            self.matrices[sid] = norm_mat
            mean_emb = np.mean(norm_mat, axis=0)
            c_norm = np.linalg.norm(mean_emb)
            self.centroids[sid] = mean_emb / (c_norm if c_norm > 0 else 1.0)

    def identify(self, embedding: np.ndarray, duration: float, current_time: float, threshold_override=None):
        target = l2_normalize(embedding.astype(np.float32))
        thresh = threshold_override if threshold_override is not None else self.threshold

        if not self.speakers:
            new_id = 1
            color = self.colors[0]
            self.speakers[new_id] = {
                "name": f"Speaker {new_id}",
                "color": color,
                "embeddings": [target]
            }
            self.last_speaker_id = new_id
            self.last_speaker_time = current_time
            self._rebuild_cache()
            return new_id, f"Speaker {new_id}", color, 1.0

        best_id = None
        best_score = -1.0

        for sid, centroid in self.centroids.items():
            centroid_score = float(np.dot(centroid, target))
            mat = self.matrices.get(sid)
            if mat is not None and len(mat) > 0:
                scores = np.dot(mat, target)
                k = min(3, len(scores))
                top_k = np.partition(scores, -k)[-k:]
                top_avg = float(np.mean(top_k))
                max_s = float(np.max(scores))
                score = max(centroid_score, top_avg * 0.85 + max_s * 0.15)
            else:
                score = centroid_score

            if self.last_speaker_id == sid and (current_time - self.last_speaker_time) < 8.0:
                score += 0.04

            if score > best_score:
                best_score = score
                best_id = sid

        if best_id is not None and best_score >= thresh:
            if duration >= 0.8 and len(self.speakers[best_id]["embeddings"]) < 50:
                self.speakers[best_id]["embeddings"].append(target)
                self._rebuild_cache()
            self.last_speaker_id = best_id
            self.last_speaker_time = current_time
            info = self.speakers[best_id]
            return best_id, info["name"], info["color"], best_score
        else:
            new_id = len(self.speakers) + 1
            color = self.colors[(new_id - 1) % len(self.colors)]
            self.speakers[new_id] = {
                "name": f"Speaker {new_id}",
                "color": color,
                "embeddings": [target]
            }
            self.last_speaker_id = new_id
            self.last_speaker_time = current_time
            self._rebuild_cache()
            return new_id, f"Speaker {new_id}", color, max(0.0, best_score)

def extract_speaker_embedding(audio_segment: np.ndarray, embedder_model, dev_type="cuda") -> np.ndarray:
    if len(audio_segment) < 9600:
        pad_len = 12800 - len(audio_segment)
        audio_segment = np.pad(audio_segment, (0, pad_len), mode='constant')
    with torch.inference_mode():
        w_t = torch.from_numpy(audio_segment).float().unsqueeze(0).to(dev_type)
        e = embedder_model.encode_batch(w_t).squeeze().detach().cpu().numpy()
        return l2_normalize(e)

# 4. Urdu Text Normalizer & Roman Urdu Transliteration
class UrduNormalizer:
    CHAR_MAP = {
        '\u064A': '\u06CC', '\u0649': '\u06CC', '\u0626': '\u06C1',
        '\u0643': '\u06A9', '\u0629': '\u06C3', '\u0647': '\u06C1',
        '\u0624': '\u0648', '\u060C': '،', '\u061B': '؛', '\u061F': '؟',
        '\u06F0': '0', '\u06F1': '1', '\u06F2': '2', '\u06F3': '3', '\u06F4': '4',
        '\u06F5': '5', '\u06F6': '6', '\u06F7': '7', '\u06F8': '8', '\u06F9': '9',
    }
    DIACRITICS = re.compile(r'[\u064B-\u065F\u0670\u06D6-\u06ED]')
    EXTRA_SPACES = re.compile(r'\s+')

    @classmethod
    def normalize(cls, text: str) -> str:
        if not text: return ""
        for ar, ur in cls.CHAR_MAP.items():
            text = text.replace(ar, ur)
        text = cls.DIACRITICS.sub('', text)
        return cls.EXTRA_SPACES.sub(' ', text).strip()

URDU_TO_ROMAN_DICT = {
    "کیا": "kya", "ہے": "hai", "ہیں": "hain", "کا": "ka", "کی": "ki", "کے": "ke",
    "کو": "ko", "سے": "se", "میں": "mein", "پر": "par", "اور": "aur", "یہ": "yeh",
    "وہ": "woh", "نہیں": "nahi", "تو": "toh", "بھی": "bhi", "کر": "kar", "رہا": "raha",
    "رہی": "rahi", "رہے": "rahe", "تھا": "tha", "تھی": "thi", "تھے": "the", "ہوں": "hoon",
    "گے": "gay", "گا": "ga", "گی": "gi", "نے": "ne", "آپ": "aap", "ہم": "hum",
    "تم": "tum", "ایک": "aik", "دو": "do", "تین": "teen", "چار": "chaar", "پانچ": "paanch",
    "بہت": "bohot", "اچھا": "acha", "صحیح": "sahi", "بات": "baat", "کام": "kaam",
    "سب": "sab", "کچھ": "kuch", "کون": "kaun", "کب": "kab", "کہاں": "kahan",
    "کیسے": "kaise", "کیوں": "kyun", "اگر": "agar", "مگر": "magar", "لیکن": "lekin"
}

def to_roman_urdu(text: str) -> str:
    words = text.split()
    converted = [URDU_TO_ROMAN_DICT.get(w, w) for w in words]
    return " ".join(converted)

def post_process_english(text: str) -> str:
    if not text: return ""
    text = re.sub(r'\s+([,.?!;:،؟])', r'\1', text)
    text = re.sub(r'([.?!])\s*([a-z])', lambda m: f"{m.group(1)} {m.group(2).upper()}", text)
    text = text.strip()
    if text and text[0].islower():
        text = text[0].upper() + text[1:]
    return text

RE_HAL_REPEATS = re.compile(r'(\b\w+\b)(?:\s+\1){3,}', flags=re.IGNORECASE)
def sanitize_script(text: str) -> str:
    if not text: return ""
    for noise in ["Subtitles by", "Amara.org", "Thank you for watching", "Please subscribe", "Transcribed by"]:
        text = text.replace(noise, "")
    return text.strip()

# 5. Configurations
MODEL_MAP = {
    'Large-v3-Turbo (809M)': 'large-v3-turbo',
    'Whisper Large-v3 (1.5B)': 'large-v3',
    'Whisper Base (74M)': 'base',
    'Whisper Tiny (39M)': 'tiny',
    'Whisper Small (244M)': 'small',
    'Whisper Medium (769M)': 'medium',
}
LANG_MAP = {
    'Bilingual (Urdu + English)': ('ur', False, False),
    'Pure Urdu Script (اردو)': ('ur', True, False),
    'English Only': ('en', False, True),
    'Roman Urdu (Latin)': ('ur', False, True),
}
PROMPT_MAP = {
    'Bilingual (Urdu + English)': "Bilingual Pakistani university classroom lecture in mixed English and Urdu. Discussion on computer science, concepts, code, formulas, assignments, presentations, questions and answers.",
    'Pure Urdu Script (اردو)': "یہ ایک پاکستانی یونیورسٹی کا کلاس روم لیکچر ہے۔ تمام الفاظ اور گفتگو کو خالص اور درست اردو رسم الخط میں تحریر کریں۔ انگریزی اصطلاحات کو بھی اردو رسم الخط میں لکھیں۔",
    'English Only': "University classroom lecture delivered strictly in English. Discussion on computer science, technical definitions, formulas, and academic questions.",
    'Roman Urdu (Latin)': "Pakistani university classroom lecture in Roman Urdu and English language. Code-switching discussion and concepts.",
}

def format_srt_time(seconds: float) -> str:
    millis = int(round((seconds - int(seconds)) * 1000))
    seconds = int(seconds)
    mins, secs = divmod(seconds, 60)
    hours, mins = divmod(mins, 60)
    return f"{hours:02d}:{mins:02d}:{secs:02d},{millis:03d}"

# 6. Audio File Loader (Mono 16kHz Float32)
def load_16k_from_file(fpath):
    if not fpath or not os.path.exists(fpath):
        return np.array([], dtype=np.float32)
    try:
        data, sr = sf.read(fpath, dtype='float32')
    except Exception:
        wav, sr = torchaudio.load(fpath)
        data = wav.numpy().squeeze()
    if data.ndim > 1:
        data = np.mean(data, axis=1 if data.shape[1] < data.shape[0] else 0)
    data = resample_to_16k(data, sr)
    return butter_highpass_filter(data)

# 7. AI Lecture Summarization & Flashcards Engine
class GeminiSummarizer:
    def __init__(self, api_key: str = None):
        self.api_key = (api_key or os.environ.get("GEMINI_API_KEY", "")).strip()

    def summarize_lecture(self, transcript_entries: list, lecture_title: str = "Classroom Lecture") -> dict:
        if not transcript_entries:
            return {"success": False, "message": "No transcript content available to summarize. Please record or upload speech first."}

        if not self.api_key:
            return {
                "success": False,
                "message": "AI API Key is required for lecture summary. Please enter your API key in the sidebar."
            }

        lines = []
        for e in transcript_entries:
            spk = e.get("speaker", e.get("speaker_name", "Speaker"))
            time_str = e.get("time", e.get("timestamp", ""))
            text = e.get("text", "").strip()
            if text:
                prefix = f"[{time_str}] {spk}: " if time_str else f"{spk}: "
                lines.append(f"{prefix}{text}")

        full_transcript = "\n".join(lines)
        if len(full_transcript) < 30:
            return {"success": False, "message": "Transcript is too short to generate a meaningful summary."}

        system_instruction = (
            "You are an expert academic AI note-taker and university study assistant for 'VoiceDiary'. "
            "Analyze classroom lecture transcripts (including mixed English and Urdu code-switching) "
            "and produce clean, beautifully structured study notes in standard GitHub-flavored Markdown.\n\n"
            "CRITICAL FORMATTING GUIDELINES:\n"
            "- Start directly with the lecture title heading: # [Lecture Title]\n"
            "- Do NOT include conversational intro/outro filler (never say 'Here are your notes', do NOT write horizontal rules '---' under the title).\n"
            "- Never write negative disclaimers like '(No mathematical formulas were mentioned)'. Focus purely on what was taught.\n"
            "- For Review Questions, format each question and answer clearly as:\n"
            "  * **Q:** [Clear question text]?\n"
            "    **A:** [Concise, accurate answer text]\n\n"
            "REQUIRED SECTIONS:\n"
            "## 📌 Executive Lecture Summary\n"
            "(3-4 high-yield bullet points summarizing core themes)\n\n"
            "## 🎯 Key Concepts & Academic Definitions\n"
            "(In-depth breakdown of concepts, formulas, logic, or technical steps discussed)\n\n"
            "## 💡 Key Review Questions\n"
            "(4-5 high-yield Question and Answer pairs for rapid exam revision)\n\n"
            "## 📋 Action Items & Homework Mentioned\n"
            "(Any tasks, assignments, deadlines, or follow-ups mentioned, or 'None mentioned' if none)\n\n"
            "If any technical terms were spoken in Roman Urdu, explain the concept clearly in English and Urdu."
        )

        user_prompt = f"Lecture Title: {lecture_title}\n\nTranscript Content:\n{full_transcript}"
        url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={self.api_key}"

        payload = {
            "contents": [
                {
                    "parts": [
                        {"text": f"{system_instruction}\n\n{user_prompt}"}
                    ]
                }
            ],
            "generationConfig": {
                "temperature": 0.25,
                "topP": 0.95,
                "maxOutputTokens": 2048,
            }
        }

        try:
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={"Content-Type": "application/json"}
            )

            with urllib.request.urlopen(req, timeout=30) as response:
                res_data = json.loads(response.read().decode("utf-8"))
                candidates = res_data.get("candidates", [])
                if not candidates:
                    return {"success": False, "message": "AI service returned an empty response."}

                raw_summary = candidates[0].get("content", {}).get("parts", [{}])[0].get("text", "")
                clean_text = re.sub(r'^(?:Here are (?:the|your)|Sure,|Below is|Here is).*?$\n?', '', raw_summary.strip(), flags=re.MULTILINE | re.IGNORECASE)
                clean_text = re.sub(r'\(No (?:mathematical|formulas|code).*?\)', '', clean_text, flags=re.IGNORECASE)
                clean_text = re.sub(r'(#[^\n]+\n+)\s*---\s*\n+', r'\1', clean_text)
                clean_text = re.sub(r'\n{3,}', '\n\n', clean_text).strip()

                return {
                    "success": True,
                    "summary_markdown": clean_text,
                    "model": "gemini-2.5-flash"
                }

        except urllib.error.HTTPError as he:
            err_msg = he.read().decode("utf-8", errors="ignore")
            return {"success": False, "message": f"API Error ({he.code}): {err_msg}"}
        except Exception as ex:
            return {"success": False, "message": f"Failed to generate summary: {str(ex)}"}

def render_summary_to_html(md_text: str) -> str:
    header_html = """<div class="vd-sec-title">
      <span>Lecture Summary &amp; Key Notes</span>
    </div>"""

    if not md_text:
        return f"""{header_html}
        <div class="vd-window-box vd-empty-box">
          <div class="vd-empty-icon">✨</div>
          <div class="vd-empty-title">No Lecture Summary Generated Yet</div>
          <div class="vd-empty-sub">Click '✨ Generate Lecture Summary' in the sidebar after recording or transcribing</div>
        </div>"""

    text = md_text.strip()
    text = re.sub(r'^(?:Here are (?:the|your)|Sure,|Below is|Here is|Note:).*?$\n?', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'\(No (?:mathematical|formulas|code).*?\)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'(#[^\n]+\n+)\s*---\s*\n+', r'\1', text)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()

    flashcard_raw_pattern = re.compile(
        r'^\s*[\*\-•]?\s*\*{0,2}Q:?\*{0,2}\s*(.*?)(?:\n\s*|\s*—\s*)\s*\*{0,2}A:?\*{0,2}\s*(.*?)$',
        flags=re.MULTILINE
    )
    text = flashcard_raw_pattern.sub(
        r'___VD_FLASHCARD_START___\1___VD_FLASHCARD_MID___\2___VD_FLASHCARD_END___',
        text
    )

    html = text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')

    html = re.sub(r'^\s*---\s*$', r'<hr class="vd-summary-hr">', html, flags=re.MULTILINE)
    html = re.sub(r'^### (.*$)', r'<h4 class="vd-sum-h4">\1</h4>', html, flags=re.MULTILINE)
    html = re.sub(r'^## (.*$)', r'<h3 class="vd-sum-h3">\1</h3>', html, flags=re.MULTILINE)
    html = re.sub(r'^# (.*$)', r'<h2 class="vd-sum-h2">\1</h2>', html, flags=re.MULTILINE)

    html = re.sub(r'\*\*\*(.*?)\*\*\*', r'<strong><em>\1</em></strong>', html)
    html = re.sub(r'\*\*(.*?)\*\*', r'<strong class="vd-sum-strong">\1</strong>', html)
    html = re.sub(r'\*(.*?)\*', r'<em class="vd-sum-em">\1</em>', html)
    html = re.sub(r'`([^`]+)`', r'<code class="vd-sum-code">\1</code>', html)

    html = re.sub(
        r'___VD_FLASHCARD_START___(.*?)___VD_FLASHCARD_MID___(.*?)___VD_FLASHCARD_END___',
        r'<div class="vd-flashcard"><div class="vd-flashcard-q">💡 Question: \1</div><div class="vd-flashcard-a"><strong>Answer:</strong> \2</div></div>',
        html,
        flags=re.DOTALL
    )

    html = re.sub(r'^\s*[\*\-•]\s+(.*$)', r'<li class="vd-sum-li">\1</li>', html, flags=re.MULTILINE)
    html = re.sub(r'^\s*\d+\.\s+(.*$)', r'<li class="vd-sum-li">\1</li>', html, flags=re.MULTILINE)
    html = re.sub(r'(<li class="vd-sum-li">[\s\S]*?</li>(?:\s*<li class="vd-sum-li">[\s\S]*?</li>)*)', r'<ul class="vd-sum-ul">\1</ul>', html)

    paragraphs = []
    for p in html.split('\n\n'):
        p = p.strip()
        if not p: continue
        if p.startswith('<h') or p.startswith('<ul') or p.startswith('<div') or p.startswith('<hr'):
            paragraphs.append(p)
        else:
            paragraphs.append(f'<p class="vd-sum-p">{p}</p>')

    rendered = "".join(paragraphs)
    return f"""{header_html}
    <div class="vd-window-box vd-summary-window">
      <div class="vd-summary-toolbar">
        <div class="vd-summary-badge">Executive Summary</div>
        <button class="vd-tool-btn" onclick="window.copySummaryText()">📋 Copy Summary</button>
      </div>
      <div class="vd-summary-body" id="vd-summary-text-container">{rendered}</div>
    </div>"""

def render_summary_loading_html() -> str:
    header_html = """<div class="vd-sec-title">
      <span>Lecture Summary &amp; Key Notes</span>
      <span class="vd-sec-meta" style="background:rgba(99,102,241,0.15); color:#818CF8; border:1px solid rgba(99,102,241,0.3);">⚡ AI Synthesizing...</span>
    </div>"""

    return f"""{header_html}
    <div class="vd-window-box vd-summary-window" style="min-height: 260px; display: flex; flex-direction: column; align-items: center; justify-content: center; text-align: center; gap: 16px; padding: 40px 20px;">
      <div class="vd-ai-pulse-wrapper">
        <div class="vd-ai-spinner"></div>
        <div class="vd-ai-sparkle">✨</div>
      </div>
      <div style="max-width: 480px;">
        <div style="font-size: 16.5px; font-weight: 700; color: #FFFFFF; margin-bottom: 6px; letter-spacing: -0.01em;">
          Synthesizing Lecture...
        </div>
        <div style="font-size: 13.5px; color: #94A3B8; line-height: 1.5;">
          Extracting key lecture takeaways, action items, executive overview &amp; exam flashcards
        </div>
      </div>
      <div class="vd-loading-shimmer-bar">
        <div class="vd-loading-shimmer-progress"></div>
      </div>
    </div>"""

# 8. Render UI Components & 5-Format Exporters
def render_transcript_ui(segments_list, matcher_instance, mkey, elapsed_total, summary_md=""):
    header_transcript = f"""<div class="vd-sec-title">
      <span>Classroom Transcript &amp; Lecture Notes</span>
      <span class="vd-sec-meta">{len(segments_list)} segments</span>
    </div>"""

    header_speakers = f"""<div class="vd-sec-title">
      <span>Speakers</span>
      <span class="vd-sec-meta">{len(matcher_instance.speakers)} active</span>
    </div>"""

    header_exports = """<div class="vd-sec-title">
      <span>Export &amp; Download Notes</span>
    </div>"""

    if not segments_list:
        empty_t = f"""{header_transcript}
        <div class="vd-window-box vd-empty-box" style="height: 380px;">
          <div class="vd-empty-icon">🎙️</div>
          <div class="vd-empty-title">Live Classroom Transcript Will Stream Here</div>
          <div class="vd-empty-sub">Click 'Record' above or upload an audio file to start transcribing</div>
        </div>"""

        empty_s = f"""{header_speakers}
        <div class="vd-window-box vd-empty-box" style="height: 140px;">
          <div class="vd-empty-title" style="font-size:13px;">No Active Speakers</div>
          <div class="vd-empty-sub">Start speaking to build neural voiceprints</div>
        </div>"""

        empty_e = f"""{header_exports}
        <div class="vd-export-grid">
          <div class="vd-export-card disabled">
            <span class="vd-export-icon">📝</span>
            <div class="vd-export-name">Markdown (.md)</div>
            <div class="vd-export-desc">0 segments</div>
          </div>
          <div class="vd-export-card disabled">
            <span class="vd-export-icon">📄</span>
            <div class="vd-export-name">Plain Text (.txt)</div>
            <div class="vd-export-desc">0 segments</div>
          </div>
          <div class="vd-export-card disabled">
            <span class="vd-export-icon">🎬</span>
            <div class="vd-export-name">Subtitles (.srt)</div>
            <div class="vd-export-desc">0 segments</div>
          </div>
          <div class="vd-export-card disabled">
            <span class="vd-export-icon">📦</span>
            <div class="vd-export-name">JSON (.json)</div>
            <div class="vd-export-desc">0 segments</div>
          </div>
          <div class="vd-export-card disabled">
            <span class="vd-export-icon">✨</span>
            <div class="vd-export-name">Summary (.md)</div>
            <div class="vd-export-desc">Generate in sidebar</div>
          </div>
        </div>"""
        return empty_t, empty_s, empty_e, ""

    cards_html = []
    plain_lines = []
    srt_entries = []

    for i, seg in enumerate(segments_list, 1):
        sid = seg["id"]
        sname = seg["speaker"]
        stime = seg.get("time", "00:00")
        stext = seg.get("text", "")
        scolor = seg.get("color", "#6366F1")
        sstart = seg.get("start", 0.0)
        send = seg.get("end", sstart + 1.0)
        initial = sname[0].upper() if sname else "S"

        plain_lines.append(f"[{stime}] {sname}: {stext}")
        srt_entries.append(f"{i}\n{format_srt_time(sstart)} --> {format_srt_time(send)}\n[{sname}] {stext}\n")

        cards_html.append(f"""
        <div class="vd-seg-card" data-spk-seg="{sid}" style="border-left: 3.5px solid {scolor};">
          <div class="vd-seg-header">
            <div class="vd-seg-speaker">
              <span class="vd-seg-avatar" style="background:{scolor}22; color:{scolor}; border:1px solid {scolor}55;">{initial}</span>
              <span class="vd-spk-lbl" style="color:{scolor};">{sname}</span>
            </div>
            <span class="vd-seg-time">{stime}</span>
          </div>
          <div class="vd-seg-body">{stext}</div>
        </div>
        """)

    spk_cards_html = []
    for sid, sdata in matcher_instance.speakers.items():
        sname = sdata["name"]
        scolor = sdata["color"]
        cnt = len(sdata["embeddings"])
        initial = sname[0].upper() if sname else "S"
        spk_cards_html.append(f"""
        <div class="vd-spk-card" data-spk-card="{sid}" style="border-left: 3px solid {scolor};">
          <div class="vd-spk-left">
            <div class="vd-spk-av" style="background:{scolor}22; color:{scolor}; border:1px solid {scolor}55;">{initial}</div>
            <div>
              <div class="vd-spk-name">{sname}</div>
              <div class="vd-spk-meta">{cnt} voice samples</div>
            </div>
          </div>
          <button class="vd-spk-edit-btn" onclick="window.renameSpeaker('{sid}', '{sname}')">
            Edit
          </button>
        </div>
        """)

    # 1. Markdown
    md_content = f"# VoiceDiary Lecture Notes\n**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n**Model:** {mkey}\n**Processing Time:** {elapsed_total:.2f}s\n\n## Transcript\n\n"
    for p in plain_lines:
        md_content += f"{p}\n\n"
    md_b64 = base64.b64encode(md_content.encode("utf-8")).decode("ascii")
    md_href = f"data:text/markdown;charset=utf-8;base64,{md_b64}"

    # 2. Plain Text
    txt_content = f"VoiceDiary Lecture Transcript\nGenerated: {time.strftime('%Y-%m-%d %H:%M:%S')}\nModel: {mkey}\nProcessing Time: {elapsed_total:.2f}s\n\n" + "\n".join(plain_lines)
    txt_b64 = base64.b64encode(txt_content.encode("utf-8")).decode("ascii")
    txt_href = f"data:text/plain;charset=utf-8;base64,{txt_b64}"

    # 3. Subtitles SRT
    srt_content = "\n".join(srt_entries)
    srt_b64 = base64.b64encode(srt_content.encode("utf-8")).decode("ascii")
    srt_href = f"data:text/plain;charset=utf-8;base64,{srt_b64}"

    # 4. JSON
    json_payload = {
        "metadata": {
            "title": "VoiceDiary Lecture Transcript",
            "timestamp": time.strftime('%Y-%m-%d %H:%M:%S'),
            "model": mkey,
            "processing_time_sec": elapsed_total
        },
        "speakers": {sid: sdata["name"] for sid, sdata in matcher_instance.speakers.items()},
        "segments": segments_list
    }
    json_b64 = base64.b64encode(json.dumps(json_payload, indent=2, ensure_ascii=False).encode("utf-8")).decode("ascii")
    json_href = f"data:application/json;charset=utf-8;base64,{json_b64}"

    # 5. AI Summary (if present)
    if summary_md:
        sum_b64 = base64.b64encode(summary_md.encode("utf-8")).decode("ascii")
        sum_href = f"data:text/markdown;charset=utf-8;base64,{sum_b64}"
        sum_card = f"""
        <a href="{sum_href}" download="voicediary_lecture_summary.md" class="vd-export-card" style="text-decoration:none; border-color:rgba(245,158,11,0.5);">
          <span class="vd-export-icon">✨</span>
          <div class="vd-export-name" style="color:#FBBF24;">Summary (.md)</div>
          <div class="vd-export-desc">Executive notes &amp; key points</div>
        </a>"""
    else:
        sum_card = """
        <div class="vd-export-card disabled">
          <span class="vd-export-icon">✨</span>
          <div class="vd-export-name">Summary (.md)</div>
          <div class="vd-export-desc">Generate via sidebar</div>
        </div>"""

    export_html = f"""{header_exports}
    <div class="vd-export-grid">
      <a href="{md_href}" download="voicediary_lecture.md" class="vd-export-card" style="text-decoration:none;">
        <span class="vd-export-icon">📝</span>
        <div class="vd-export-name">Markdown (.md)</div>
        <div class="vd-export-desc">{len(segments_list)} segments</div>
      </a>
      <a href="{txt_href}" download="voicediary_lecture.txt" class="vd-export-card" style="text-decoration:none;">
        <span class="vd-export-icon">📄</span>
        <div class="vd-export-name">Plain Text (.txt)</div>
        <div class="vd-export-desc">Full text transcript</div>
      </a>
      <a href="{srt_href}" download="voicediary_lecture.srt" class="vd-export-card" style="text-decoration:none;">
        <span class="vd-export-icon">🎬</span>
        <div class="vd-export-name">Subtitles (.srt)</div>
        <div class="vd-export-desc">Timed subtitle track</div>
      </a>
      <a href="{json_href}" download="voicediary_lecture.json" class="vd-export-card" style="text-decoration:none;">
        <span class="vd-export-icon">📦</span>
        <div class="vd-export-name">JSON (.json)</div>
        <div class="vd-export-desc">Structured speaker data</div>
      </a>
      {sum_card}
    </div>"""

    transcript_feed_html = f"""{header_transcript}
    <div class="vd-window-box vd-transcript-window" id="vd-transcript-feed">
      {"\n".join(cards_html)}
    </div>"""

    speakers_feed_html = f"""{header_speakers}
    <div class="vd-window-box vd-speaker-window">
      {"\n".join(spk_cards_html)}
    </div>"""

    return transcript_feed_html, speakers_feed_html, export_html, "\n".join(plain_lines)


In [ ]:
# Step 4: Launch Complete Web Interface

CSS = """
:root {
  --vd-bg: #070B14;
  --vd-card-bg: #0F172A;
  --vd-card-sub: #131C2E;
  --vd-window-bg: #0A0F1D;
  --vd-border: rgba(255, 255, 255, 0.08);
  --vd-border-input: rgba(255, 255, 255, 0.12);
  --vd-text-primary: #FFFFFF;
  --vd-text-secondary: #94A3B8;
  --vd-text-muted: #64748B;
  --vd-accent: #6366F1;
  --vd-accent-hover: #4F46E5;
}

body, .gradio-container {
  background-color: var(--vd-bg) !important;
  color: var(--vd-text-primary) !important;
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Inter', Roboto, sans-serif !important;
  font-size: 16px !important;
  line-height: 1.55 !important;
  -webkit-font-smoothing: antialiased !important;
}

/* Full Width Spacious Dashboard */
.gradio-container {
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  padding: 22px 32px !important;
  box-sizing: border-box !important;
}

.gradio-container > .main,
.gradio-container > .main > .wrap,
.gradio-container .contain,
.gradio-container .main {
  max-width: 100% !important;
  width: 100% !important;
  padding: 0 !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}

/* Eliminate duplicate loading spinner overlays */
.pending, .loading, .progress-text, .meta-text, .eta-bar, .wrap.default.full,
.gradio-container .progress-text, .gradio-container .meta-text, .gradio-container .pending {
  display: none !important;
  visibility: hidden !important;
  opacity: 0 !important;
}

/* Spacious Layout Grid & Columns */
.gradio-container .row,
.gradio-container .gap-6 {
  gap: 28px !important;
  margin: 0 !important;
  width: 100% !important;
  box-sizing: border-box !important;
  align-items: stretch !important;
}

.gradio-container .column {
  padding: 0 !important;
  margin: 0 !important;
  box-sizing: border-box !important;
  display: flex !important;
  flex-direction: column !important;
  gap: 18px !important;
  overflow-x: hidden !important;
}

/* Universal Horizontal Scrollbar Killer */
.gradio-container,
.gradio-container *,
.gradio-container .column,
.gradio-container .column *,
.gradio-container .gradio-accordion,
.gradio-container .gradio-accordion *,
.gradio-container details,
.gradio-container details *,
.gradio-container .slider-container,
.gradio-container .gradio-slider,
.gradio-container .range_wrap,
.gradio-container input[type=range] {
  overflow-x: hidden !important;
  scrollbar-width: none !important;
  -ms-overflow-style: none !important;
  min-width: 0 !important;
}

.gradio-container::-webkit-scrollbar,
.gradio-container *::-webkit-scrollbar,
.gradio-container details::-webkit-scrollbar,
.gradio-container details *::-webkit-scrollbar,
.gradio-container .gradio-accordion::-webkit-scrollbar,
.gradio-container .gradio-accordion *::-webkit-scrollbar {
  display: none !important;
  width: 0 !important;
  height: 0 !important;
}

/* Strip outer wrapper container borders/backgrounds */
.gradio-container .block,
.gradio-container [data-testid="block-container"],
.gradio-container .form,
.gradio-container .panel,
.gradio-container .component,
.gradio-container .gr-form,
.gradio-container .gr-box,
.gradio-container .gr-panel,
.gradio-container .gr-block-wrapper,
.gradio-container .block.gradio-html,
.gradio-container .html,
.gradio-container .prose,
.gradio-container fieldset,
.gradio-container .tabitem,
.gradio-container .tabitem > div,
.gradio-container .tabitem .row {
  background: transparent !important;
  border: none !important;
  box-shadow: none !important;
  margin-left: 0 !important;
  margin-right: 0 !important;
  padding: 0 !important;
  width: 100% !important;
  box-sizing: border-box !important;
}

/* ── REMOVE GAP BELOW HEADINGS (RED MARKS) ── */
.gradio-container .block.gradio-html:has(.vd-sec-title) {
  margin-bottom: -14px !important;
  margin-top: 6px !important;
  padding: 0 !important;
}

.gradio-container .block.gradio-html:first-child {
  margin-top: 0 !important;
  margin-bottom: -14px !important;
}

/* ── ADD GAP BETWEEN COMPONENTS (GREEN MARKS) ── */
/* Gap below Dropdowns (Matches 20px Slider Spacing) */
.gradio-container .gradio-dropdown,
.gradio-container .block.gradio-dropdown,
.gradio-container [data-testid="dropdown"],
.gradio-container .vd-dropdown {
  margin-top: 0 !important;
  margin-bottom: 20px !important;
  padding: 0 !important;
}

/* Textbox */
.gradio-container .block.gradio-textbox {
  margin-bottom: 12px !important;
  margin-top: 0 !important;
  padding: 0 !important;
}

/* Clean & balanced spacing between sliders */
.gradio-container .block.gradio-slider,
.gradio-container .gradio-slider {
  margin-top: 0 !important;
  margin-bottom: 20px !important;
  padding: 0 !important;
}

.gradio-container .block.gradio-slider:first-child,
.gradio-container .gradio-slider:first-child {
  margin-top: 4px !important;
}

.gradio-container .block.gradio-slider:last-child,
.gradio-container .gradio-slider:last-child,
.gradio-container .block.gradio-slider:last-of-type,
.gradio-container .gradio-slider:last-of-type {
  margin-bottom: 4px !important;
}


/* Visual Hierarchy: Clean Large Section Titles */
.vd-sec-title {
  display: flex;
  align-items: center;
  justify-content: space-between;
  font-size: 14px;
  font-weight: 700;
  text-transform: uppercase;
  letter-spacing: 0.08em;
  color: #818CF8;
  margin: 0 0 2px 0 !important;
  padding: 0 !important;
  width: 100%;
  box-sizing: border-box;
  text-align: left;
  line-height: 1.2;
}
.vd-sec-meta {
  font-size: 13px;
  font-weight: 600;
  text-transform: none;
  letter-spacing: 0;
  color: var(--vd-text-secondary);
  background: rgba(255,255,255,0.06);
  padding: 3px 10px;
  border-radius: 9999px;
  display: inline-flex;
  align-items: center;
  justify-content: center;
  line-height: 1;
}

/* Visual Hierarchy: Parameter / Sub-Headings */
.gradio-container label > span,
.gradio-container label span,
.gradio-container .label-wrap span,
.gradio-container .block > label > span {
  font-size: 13.5px !important;
  font-weight: 600 !important;
  color: #E2E8F0 !important;
  line-height: 1.3 !important;
  display: block !important;
  margin-top: 0 !important;
  margin-bottom: 4px !important;
  padding-left: 0 !important;
  text-align: left !important;
  letter-spacing: 0.01em !important;
}

/* Visual Hierarchy: Helper / Info Descriptions (Sub-text) */
.gradio-container .info,
.gradio-container .description,
.gradio-container .help-text,
.gradio-container p.info,
.gradio-container span.info {
  font-size: 12px !important;
  color: #94A3B8 !important;
  line-height: 1.35 !important;
  text-align: left !important;
  display: block !important;
  margin-top: 2px !important;
  margin-bottom: 6px !important;
  font-weight: 400 !important;
  padding-left: 0 !important;
}

/* Large Clean Dropdowns */
.gradio-container .gradio-dropdown .wrap,
.gradio-container .gradio-dropdown [class*="wrap"] {
  background: var(--vd-card-sub) !important;
  border: 1px solid var(--vd-border-input) !important;
  border-radius: 8px !important;
  padding: 2px 10px !important;
  min-height: 48px !important;
  box-sizing: border-box !important;
  box-shadow: none !important;
  width: 100% !important;
  margin: 0 !important;
  transition: border-color 0.15s ease, box-shadow 0.15s ease !important;
}

.gradio-container .gradio-dropdown:focus-within .wrap,
.gradio-container .gradio-dropdown:focus-within [class*="wrap"] {
  border-color: var(--vd-accent) !important;
  box-shadow: 0 0 0 2px rgba(99, 102, 241, 0.25) !important;
}

.gradio-container .gradio-dropdown input,
.gradio-container .gradio-dropdown .wrap-inner,
.gradio-container .gradio-dropdown select,
.gradio-container .gradio-dropdown .secondary-wrap {
  background: transparent !important;
  border: none !important;
  box-shadow: none !important;
  font-size: 16px !important;
  color: #FFFFFF !important;
  padding: 10px 6px !important;
  text-align: left !important;
  margin: 0 !important;
}

.gradio-container select option {
  background: #131C2E !important;
  color: #FFFFFF !important;
  font-size: 15.5px !important;
}

/* Large Clean Textboxes */
.gradio-container .gradio-textbox input,
.gradio-container .gradio-textbox textarea {
  background-color: var(--vd-card-sub) !important;
  border: 1px solid var(--vd-border-input) !important;
  border-radius: 8px !important;
  font-size: 16px !important;
  color: #FFFFFF !important;
  padding: 12px 16px !important;
  min-height: 48px !important;
  width: 100% !important;
  box-sizing: border-box !important;
  text-align: left !important;
  box-shadow: none !important;
  margin: 0 !important;
  transition: border-color 0.15s ease, box-shadow 0.15s ease !important;
}

.gradio-container .gradio-textbox:focus-within input,
.gradio-container .gradio-textbox:focus-within textarea {
  border-color: var(--vd-accent) !important;
  box-shadow: 0 0 0 2px rgba(99, 102, 241, 0.25) !important;
  outline: none !important;
}

/* Accordion & Sensitivity Tuning - Clean Main Section Card */
.gradio-container details {
  background: var(--vd-card-bg) !important;
  border: 1px solid var(--vd-border) !important;
  border-radius: 8px !important;
  padding: 12px 18px 18px 18px !important;
  margin-top: 10px !important;
  margin-bottom: 0 !important;
  width: 100% !important;
  max-width: 100% !important;
  box-sizing: border-box !important;
  overflow: hidden !important;
}

.gradio-container details summary {
  padding: 4px 0 !important;
  cursor: pointer !important;
  font-size: 15px !important;
  font-weight: 700 !important;
  color: #A5B4FC !important;
  display: flex !important;
  align-items: center !important;
  overflow: hidden !important;
  letter-spacing: 0.01em !important;
  transition: color 0.15s ease !important;
}

.gradio-container details[open] summary {
  padding-bottom: 10px !important;
  margin-bottom: 14px !important;
  border-bottom: 1px solid rgba(255, 255, 255, 0.08) !important;
}

.gradio-container details summary:hover {
  color: #FFFFFF !important;
}

/* Slider Number Inputs - Large Centered Badge */
.gradio-container input[type=number] {
  font-size: 13.5px !important;
  min-width: 60px !important;
  width: auto !important;
  text-align: center !important;
  padding: 4px 10px !important;
  line-height: 1.2 !important;
  -moz-appearance: textfield !important;
  appearance: textfield !important;
  display: inline-flex !important;
  align-items: center !important;
  justify-content: center !important;
  background-color: var(--vd-card-sub) !important;
  border: 1px solid var(--vd-border-input) !important;
  border-radius: 6px !important;
  color: #FFFFFF !important;
  margin: 0 !important;
}

.gradio-container input[type=number]::-webkit-inner-spin-button,
.gradio-container input[type=number]::-webkit-outer-spin-button {
  -webkit-appearance: none !important;
  margin: 0 !important;
  display: none !important;
}

.gradio-container .head button,
.gradio-container .slider-container button,
.gradio-container input[type=number] ~ button,
.gradio-container input[type=number] + button {
  display: none !important;
}

/* Slider Range Track & Min/Max Numbers */
.gradio-container .range_wrap {
  margin-top: 4px !important;
  margin-bottom: 6px !important;
}

.gradio-container .range_wrap span,
.gradio-container .slider-container span {
  font-size: 13.5px !important;
  color: #94A3B8 !important;
}

/* Header */
.vd-hdr {
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 16px 24px;
  background: var(--vd-card-bg);
  border: 1px solid var(--vd-border);
  border-radius: 10px;
  margin-bottom: 20px;
  width: 100%;
  box-sizing: border-box;
}
.vd-hdr-left { display: flex; align-items: center; gap: 16px; }
.vd-logo-box {
  width: 46px; height: 46px;
  background: linear-gradient(135deg, #6366F1, #4F46E5);
  border-radius: 9px;
  display: flex; align-items: center; justify-content: center;
  box-shadow: 0 2px 8px rgba(99,102,241,0.35);
  flex-shrink: 0;
}
.vd-hdr-title { font-size: 22px; font-weight: 700; color: #FFFFFF; letter-spacing: -0.01em; line-height: 1.2; text-align: left; }
.vd-hdr-sub { font-size: 14.5px; color: var(--vd-text-secondary); margin-top: 3px; line-height: 1.3; text-align: left; }
.vd-hw-pill {
  display: inline-flex; align-items: center; justify-content: center; gap: 8px;
  padding: 6px 14px;
  background: rgba(16,185,129,0.12);
  border: 1px solid rgba(16,185,129,0.3);
  border-radius: 9999px;
  font-size: 13px; font-weight: 600; color: #34D399;
  line-height: 1;
}
.vd-hw-dot { width: 8px; height: 8px; border-radius: 50%; background: #10B981; }

/* Dedicated Window Boxes (Scrollable & Contained) */
.vd-window-box {
  background: var(--vd-window-bg);
  border: 1px solid var(--vd-border);
  border-radius: 10px;
  padding: 14px;
  box-shadow: inset 0 1px 3px rgba(0,0,0,0.3);
  width: 100%;
  box-sizing: border-box;
}

.vd-transcript-window {
  height: 440px;
  max-height: 440px;
  overflow-y: auto;
  overflow-x: hidden;
  display: flex;
  flex-direction: column;
  gap: 12px;
  scroll-behavior: smooth;
}

.vd-speaker-window {
  max-height: 170px;
  overflow-y: auto;
  display: flex;
  flex-direction: column;
  gap: 8px;
}

.vd-summary-window {
  background: var(--vd-card-bg);
  border: 1px solid rgba(245,158,11,0.25);
  border-radius: 10px;
  padding: 22px 26px;
  width: 100%;
  box-sizing: border-box;
  text-align: left;
}

/* Empty States */
.vd-empty-box {
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
  text-align: center !important;
  color: var(--vd-text-secondary);
  width: 100%;
  box-sizing: border-box;
}
.vd-empty-icon { font-size: 30px; margin-bottom: 8px; opacity: 0.85; line-height: 1; }
.vd-empty-title { font-size: 16px; font-weight: 600; color: #E2E8F0; margin-bottom: 4px; text-align: center !important; line-height: 1.3; }
.vd-empty-sub { font-size: 14px; color: var(--vd-text-secondary); text-align: center !important; line-height: 1.4; }

/* Transcript Segment Cards */
.vd-seg-card {
  background: var(--vd-card-bg);
  border: 1px solid var(--vd-border);
  border-radius: 9px;
  padding: 14px 18px;
  flex-shrink: 0;
  width: 100%;
  box-sizing: border-box;
  text-align: left;
}
.vd-seg-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  margin-bottom: 6px;
  width: 100%;
}
.vd-seg-speaker { display: flex; align-items: center; gap: 10px; font-weight: 700; font-size: 16px; line-height: 1.2; }
.vd-seg-avatar {
  width: 28px; height: 28px;
  border-radius: 6px;
  display: inline-flex; align-items: center; justify-content: center;
  font-size: 13.5px; font-weight: 700; line-height: 1;
}
.vd-seg-time { font-size: 13.5px; color: var(--vd-text-secondary); font-family: 'JetBrains Mono', Consolas, monospace; line-height: 1; }
.vd-seg-body { font-size: 17px; line-height: 1.65; color: #FFFFFF; padding-left: 2px; text-align: left !important; word-break: break-word; }

/* Speaker Cards (Sidebar) */
.vd-spk-card {
  display: flex;
  align-items: center;
  justify-content: space-between;
  background: var(--vd-card-sub);
  border: 1px solid var(--vd-border-input);
  border-radius: 8px;
  padding: 10px 14px;
  flex-shrink: 0;
  width: 100%;
  box-sizing: border-box;
}
.vd-spk-left { display: flex; align-items: center; gap: 12px; }
.vd-spk-av {
  width: 34px; height: 34px;
  border-radius: 7px;
  display: inline-flex; align-items: center; justify-content: center;
  font-weight: 700; font-size: 14.5px; line-height: 1;
}
.vd-spk-name { font-weight: 600; font-size: 15.5px; color: #FFFFFF; text-align: left; line-height: 1.2; }
.vd-spk-meta { font-size: 13.5px; color: var(--vd-text-secondary); text-align: left; line-height: 1.2; }
.vd-spk-edit-btn {
  background: rgba(99,102,241,0.12);
  border: 1px solid rgba(99,102,241,0.28);
  color: #A5B4FC;
  border-radius: 6px;
  padding: 5px 12px;
  font-size: 13.5px;
  font-weight: 600;
  cursor: pointer;
  display: inline-flex;
  align-items: center;
  justify-content: center;
  line-height: 1;
  transition: all 0.15s ease;
}
.vd-spk-edit-btn:hover { background: #6366F1; color: #FFFFFF; }

/* Large Action Buttons */
.vd-primary-btn {
  background: linear-gradient(135deg, #6366F1 0%, #4F46E5 100%) !important;
  color: #FFFFFF !important;
  font-weight: 600 !important;
  font-size: 15.5px !important;
  border: none !important;
  border-radius: 8px !important;
  padding: 12px 20px !important;
  min-height: 46px !important;
  box-shadow: 0 2px 6px rgba(99,102,241,0.25) !important;
  transition: all 0.15s ease !important;
  width: 100% !important;
  box-sizing: border-box !important;
  display: inline-flex !important;
  align-items: center !important;
  justify-content: center !important;
  text-align: center !important;
  line-height: 1.2 !important;
  margin-top: 6px !important;
  margin-left: 0 !important;
  margin-right: 0 !important;
}
.vd-primary-btn:hover {
  filter: brightness(1.1) !important;
  transform: translateY(-1px) !important;
}

.vd-secondary-btn {
  background: rgba(255, 255, 255, 0.04) !important;
  border: 1px solid rgba(255, 255, 255, 0.1) !important;
  color: #94A3B8 !important;
  font-weight: 600 !important;
  font-size: 15px !important;
  border-radius: 8px !important;
  padding: 10px 18px !important;
  min-height: 44px !important;
  height: 44px !important;
  transition: all 0.15s ease !important;
  display: inline-flex !important;
  align-items: center !important;
  justify-content: center !important;
  text-align: center !important;
  line-height: 1.2 !important;
  width: 100% !important;
  box-sizing: border-box !important;
  margin: 0 0 14px 0 !important;
}
.vd-secondary-btn:hover {
  background: rgba(255, 255, 255, 0.08) !important;
  color: #FFFFFF !important;
  border-color: rgba(255, 255, 255, 0.2) !important;
}

/* Tabs & Top Alignment */
.gradio-container .tabs {
  margin-top: 0 !important;
  padding-top: 0 !important;
  border: none !important;
  background: transparent !important;
  width: 100% !important;
  box-sizing: border-box !important;
}

.gradio-container .tab-nav {
  margin-top: 0 !important;
  margin-bottom: 16px !important;
  border-bottom: 1px solid var(--vd-border) !important;
  display: flex !important;
  align-items: center !important;
}

.gradio-container .tab-nav button {
  font-size: 15.5px !important;
  font-weight: 600 !important;
  padding: 10px 22px !important;
  border-radius: 6px 6px 0 0 !important;
  display: inline-flex !important;
  align-items: center !important;
  justify-content: center !important;
  line-height: 1.2 !important;
}

/* Audio Recorder Component - Clean Dark Styling */
.gradio-container .audio,
.gradio-container [data-testid="audio"],
.gradio-container .audio-container {
  max-width: 100% !important;
  width: 100% !important;
  background: var(--vd-card-bg) !important;
  border: 1px solid var(--vd-border) !important;
  border-radius: 8px !important;
  box-sizing: border-box !important;
  margin: 0 0 14px 0 !important;
  padding: 10px 18px !important;
  overflow: hidden !important;
}

.gradio-container .audio select,
.gradio-container [data-testid="audio"] select {
  background-color: var(--vd-card-sub) !important;
  border: 1px solid var(--vd-border-input) !important;
  border-radius: 6px !important;
  color: #FFFFFF !important;
  font-size: 15px !important;
  padding: 8px 14px !important;
  max-width: 270px !important;
  box-sizing: border-box !important;
}

/* Hide redundant [X] clear icon inside audio recorder box */
.gradio-container .audio button[aria-label="Clear"],
.gradio-container [data-testid="audio"] button[aria-label="Clear"],
.gradio-container .audio .clear-button,
.gradio-container [data-testid="clear-button"],
.gradio-container .audio button:has(svg path[d*="M19 6.41"]) {
  display: none !important;
}

/* Live Recording Bar */
.vd-rec-bar {
  display: flex;
  align-items: center;
  justify-content: space-between;
  background: var(--vd-card-bg);
  border: 1px solid var(--vd-border);
  border-radius: 8px;
  padding: 12px 20px;
  margin-bottom: 14px;
  width: 100%;
  box-sizing: border-box;
}
.vd-rec-left {
  display: flex;
  align-items: center;
  gap: 10px;
  font-size: 15.5px;
  font-weight: 600;
  color: #E2E8F0;
  line-height: 1.2;
}
.vd-rec-dot {
  width: 10px; height: 10px; border-radius: 50%;
  background: #64748B;
  transition: all 0.2s ease;
  flex-shrink: 0;
}
.vd-rec-dot.active {
  background: #EF4444 !important;
  box-shadow: 0 0 10px #EF4444;
  animation: pulse-red 1.2s infinite;
}
@keyframes pulse-red {
  0% { transform: scale(0.95); opacity: 0.85; }
  50% { transform: scale(1.25); opacity: 1; }
  100% { transform: scale(0.95); opacity: 0.85; }
}
.vd-rec-timer {
  font-family: 'JetBrains Mono', Consolas, monospace;
  font-size: 15.5px;
  font-weight: 700;
  color: #A5B4FC;
  background: rgba(99,102,241,0.12);
  padding: 4px 12px;
  border-radius: 5px;
  border: 1px solid rgba(99,102,241,0.25);
  display: inline-flex;
  align-items: center;
  justify-content: center;
  line-height: 1;
}

/* Summary Toolbar & Typography */
.vd-summary-toolbar {
  display: flex;
  align-items: center;
  justify-content: space-between;
  margin-bottom: 16px;
  padding-bottom: 14px;
  border-bottom: 1px solid rgba(255,255,255,0.06);
  width: 100%;
  box-sizing: border-box;
}
.vd-summary-badge {
  background: rgba(245, 158, 11, 0.12);
  border: 1px solid rgba(245, 158, 11, 0.3);
  color: #FCD34D;
  padding: 5px 14px;
  border-radius: 9999px;
  font-size: 13.5px;
  font-weight: 700;
  display: inline-flex;
  align-items: center;
  line-height: 1;
}
.vd-tool-btn {
  background: rgba(99,102,241,0.12);
  border: 1px solid rgba(99,102,241,0.25);
  color: #A5B4FC;
  border-radius: 6px;
  padding: 7px 14px;
  font-size: 14px;
  font-weight: 600;
  cursor: pointer;
  display: inline-flex;
  align-items: center;
  justify-content: center;
  gap: 6px;
  line-height: 1;
}
.vd-tool-btn:hover { background: rgba(99,102,241,0.25); color: #FFFFFF; }

.vd-sum-h2 { color: #F59E0B; font-size: 20px; font-weight: 800; margin: 0 0 14px 0; border-bottom: 1.5px solid rgba(245,158,11,0.25); padding-bottom: 8px; text-align: left; }
.vd-sum-h3 { color: #FFFFFF; font-size: 17.5px; font-weight: 700; margin: 18px 0 10px 0; border-bottom: 1px solid rgba(255,255,255,0.06); padding-bottom: 5px; display: flex; align-items: center; gap: 8px; text-align: left; }
.vd-sum-h4 { color: #FBBF24; font-size: 15.5px; font-weight: 700; margin: 12px 0 6px 0; text-align: left; }
.vd-sum-strong { color: #F8FAFC; font-weight: 700; }
.vd-sum-em { color: #CBD5E1; }
.vd-sum-code { background: rgba(255,255,255,0.08); color: #A5B4FC; padding: 3px 8px; border-radius: 4px; font-family: monospace; font-size: 14px; }
.vd-sum-p { color: #E2E8F0; font-size: 16px; line-height: 1.65; margin: 8px 0; text-align: left; }
.vd-sum-ul { margin: 8px 0 14px 24px; padding: 0; text-align: left; }
.vd-sum-li { color: #E2E8F0; font-size: 16px; line-height: 1.65; margin-bottom: 5px; text-align: left; }
.vd-summary-hr { border: none; border-top: 1px solid rgba(255,255,255,0.06); margin: 18px 0; }

.vd-flashcard {
  background: rgba(99, 102, 241, 0.06);
  border: 1px solid rgba(99, 102, 241, 0.2);
  border-left: 4px solid #6366F1;
  border-radius: 8px;
  padding: 14px 18px;
  margin: 12px 0;
  width: 100%;
  box-sizing: border-box;
  text-align: left;
}
.vd-flashcard-q { color: #A5B4FC; font-weight: 700; font-size: 16px; margin-bottom: 5px; text-align: left; }
.vd-flashcard-a { color: #FFFFFF; font-size: 15.5px; line-height: 1.6; text-align: left; }

/* ── Export Grid & Cards ── */
.vd-export-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(170px, 1fr));
  gap: 14px;
  width: 100%;
  box-sizing: border-box;
}
.vd-export-card {
  background: var(--vd-card-bg);
  border: 1px solid var(--vd-border);
  border-radius: 9px;
  padding: 16px 18px;
  display: flex; flex-direction: column; gap: 4px;
  align-items: flex-start;
  text-align: left;
  transition: border-color 0.15s ease, background 0.15s ease;
  box-sizing: border-box;
}
.vd-export-card:not(.disabled):hover { border-color: #818CF8; background: #131C2E; }
.vd-export-card.disabled { opacity: 0.4; pointer-events: none; }
.vd-export-icon { font-size: 24px; line-height: 1; margin-bottom: 2px; }
.vd-export-name { font-weight: 600; font-size: 15.5px; color: #FFFFFF; text-align: left; line-height: 1.2; }
.vd-export-desc { font-size: 13.5px; color: var(--vd-text-secondary); text-align: left; line-height: 1.35; }

/* AI Summary Loading State Animation */
.vd-ai-pulse-wrapper {
  position: relative;
  width: 60px;
  height: 60px;
  display: flex;
  align-items: center;
  justify-content: center;
  margin-bottom: 2px;
}

.vd-ai-spinner {
  position: absolute;
  width: 100%;
  height: 100%;
  border-radius: 50%;
  border: 3.5px solid rgba(99, 102, 241, 0.15);
  border-top-color: #6366F1;
  border-right-color: #A855F7;
  animation: vd-spin 0.9s cubic-bezier(0.4, 0, 0.2, 1) infinite;
}

@keyframes vd-spin {
  0% { transform: rotate(0deg); }
  100% { transform: rotate(360deg); }
}

.vd-ai-sparkle {
  font-size: 24px;
  animation: vd-sparkle-pulse 1.3s ease-in-out infinite alternate;
  line-height: 1;
}

@keyframes vd-sparkle-pulse {
  0% { transform: scale(0.85); opacity: 0.75; filter: drop-shadow(0 0 4px rgba(99, 102, 241, 0.4)); }
  100% { transform: scale(1.15); opacity: 1; filter: drop-shadow(0 0 12px rgba(168, 85, 247, 0.85)); }
}

.vd-loading-shimmer-bar {
  width: 220px;
  height: 4px;
  background: rgba(255, 255, 255, 0.08);
  border-radius: 9999px;
  overflow: hidden;
  position: relative;
  margin-top: 4px;
}

.vd-loading-shimmer-progress {
  width: 50%;
  height: 100%;
  background: linear-gradient(90deg, #6366F1, #A855F7, #EC4899);
  border-radius: 9999px;
  position: absolute;
  animation: vd-shimmer 1.4s ease-in-out infinite alternate;
}

@keyframes vd-shimmer {
  0% { left: 0%; width: 30%; }
  50% { width: 60%; }
  100% { left: 70%; width: 30%; }
}

.vd-hidden-bridge {
  position: absolute !important;
  opacity: 0 !important;
  pointer-events: auto !important;
  z-index: -9999 !important;
  height: 0 !important;
  width: 0 !important;
  overflow: hidden !important;
}
"""

HEAD_JS = """
<script>
window.spkCustomNames = window.spkCustomNames || {};
window._lastRawSummary = window._lastRawSummary || "";

// ─── PROACTIVE MICROPHONE PERMISSION REQUEST ON LOAD ───
// Requests mic access on load so Gradio gets real device labels without page reload
(async function initMicPermission() {
  if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) return;
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true, video: false });
    // Stop all tracks immediately – we only needed to trigger the permission prompt
    stream.getTracks().forEach(t => t.stop());
    // Fire devicechange so Gradio re-enumerates with labelled devices
    if (navigator.mediaDevices.dispatchEvent) {
      try { navigator.mediaDevices.dispatchEvent(new Event('devicechange')); } catch(e) {}
    }
    // Small delay then sync the UI
    setTimeout(syncMicrophoneDeviceUI, 300);
    setTimeout(syncMicrophoneDeviceUI, 800);
    setTimeout(syncMicrophoneDeviceUI, 1500);
  } catch(e) {
    // Permission denied or not available – silently ignore
  }
})();

// ─── IN-PLACE MICROPHONE SYNC & ENUMERATION CONTROLLER ───
async function syncMicrophoneDeviceUI() {
  if (typeof navigator === 'undefined' || !navigator.mediaDevices || !navigator.mediaDevices.enumerateDevices) return;
  try {
    const devices = await navigator.mediaDevices.enumerateDevices();
    const audioInputs = devices.filter(d => d.kind === 'audioinput' && d.label);

    if (audioInputs.length > 0) {
      const primaryLabel = audioInputs[0].label;

      const targetEls = document.querySelectorAll('.audio *, [data-testid="audio"] *, .gradio-audio *');
      for (const el of targetEls) {
        if (el.children.length === 0 || el.tagName === 'SELECT' || el.tagName === 'OPTION') {
          const txt = (el.innerText || el.textContent || el.value || '').trim();
          if (txt.toLowerCase().includes('no microphone')) {
            if (el.tagName === 'SELECT') {
              el.innerHTML = '';
              audioInputs.forEach((dev, idx) => {
                const opt = document.createElement('option');
                opt.value = dev.deviceId;
                opt.text = dev.label || `Microphone ${idx + 1}`;
                el.appendChild(opt);
              });
              el.disabled = false;
              el.dispatchEvent(new Event('change', { bubbles: true }));
            } else if (el.tagName === 'OPTION') {
              el.text = primaryLabel;
              el.value = audioInputs[0].deviceId || '';
            } else {
              el.innerText = primaryLabel;
              el.textContent = primaryLabel;
            }
          }
        }
      }
    }
  } catch (err) {}
}

setInterval(syncMicrophoneDeviceUI, 400);
if (typeof navigator !== 'undefined' && navigator.mediaDevices && navigator.mediaDevices.addEventListener) {
  navigator.mediaDevices.addEventListener('devicechange', syncMicrophoneDeviceUI);
}

// ─── AUTO-SCROLL TRANSCRIPT WINDOW TO BOTTOM ───
let lastScrollHeight = 0;
setInterval(() => {
  const feed = document.getElementById('vd-transcript-feed');
  if (feed && feed.scrollHeight !== lastScrollHeight) {
    lastScrollHeight = feed.scrollHeight;
    feed.scrollTop = feed.scrollHeight;
  }
}, 300);

// ─── SPEAKER EDIT BUTTON CONTROLLER ───
window.renameSpeaker = function(spkId, oldName) {
  const curName = window.spkCustomNames[spkId] || oldName;
  const newName = prompt(`Enter display name for ${oldName}:`, curName);

  if (newName !== null && newName.trim() !== "") {
    const cleanName = newName.trim();
    window.spkCustomNames[spkId] = cleanName;

    document.querySelectorAll(`[data-spk-card="${spkId}"] .vd-spk-name`).forEach(el => el.innerText = cleanName);
    document.querySelectorAll(`[data-spk-card="${spkId}"] .vd-spk-av`).forEach(el => el.innerText = cleanName[0].toUpperCase());
    document.querySelectorAll(`[data-spk-seg="${spkId}"] .vd-spk-lbl`).forEach(el => el.innerText = cleanName);

    const input = document.querySelector('#hidden_rename_payload textarea, #hidden_rename_payload input');
    const btn = document.querySelector('#hidden_rename_trigger');
    if (input && btn) {
      const val = JSON.stringify({id: parseInt(spkId), name: cleanName});
      const setter = Object.getOwnPropertyDescriptor(window.HTMLTextAreaElement ? window.HTMLTextAreaElement.prototype : window.HTMLInputElement.prototype, 'value') || Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value');
      if (setter && setter.set) {
        setter.set.call(input, val);
      } else {
        input.value = val;
      }
      input.dispatchEvent(new Event('input', { bubbles: true }));
      input.dispatchEvent(new Event('change', { bubbles: true }));
      setTimeout(() => { btn.click(); }, 60);
    }
  }
};

// ─── COPY AI SUMMARY ───
window.copySummaryText = function() {
  const container = document.getElementById('vd-summary-text-container');
  const text = window._lastRawSummary || (container ? container.innerText : '');
  if (text) {
    navigator.clipboard.writeText(text).then(() => {
      alert('✅ Lecture Summary copied to clipboard!');
    }).catch(() => {
      alert('Failed to copy summary.');
    });
  } else {
    alert('No summary text available.');
  }
};

// ─── LIVE RECORDING TIMER & ROBUST STATUS CONTROLLER (ZERO RELOADS) ───
let timerInterval = null;
let recordingSeconds = 0;
let recStartTime = null;
let isRecordingActive = false;

function formatSec(s) {
  const m = Math.floor(s / 60);
  const sec = s % 60;
  return `${m.toString().padStart(2, '0')}:${sec.toString().padStart(2, '0')}`;
}

function checkIsRecording() {
  const buttons = document.querySelectorAll('button, [role="button"]');
  for (const b of buttons) {
    if (b.offsetWidth === 0 && b.offsetHeight === 0) continue;
    const txt = (b.innerText || b.textContent || '').trim().toLowerCase();
    const aria = (b.getAttribute('aria-label') || '').toLowerCase();
    const title = (b.getAttribute('title') || '').toLowerCase();
    const cls = (b.className || '').toString().toLowerCase();

    if (txt === 'stop' || txt.includes('stop recording') || txt.includes('pause recording') ||
        aria === 'stop' || aria.includes('stop recording') ||
        title === 'stop' || title.includes('stop recording') ||
        cls.includes('stop-button') || cls.includes('recording')) {
      return true;
    }
  }

  const recEls = document.querySelectorAll('.recording, [data-testid="record-pulse"], .pulse, .record-icon, [class*="recording"]');
  for (const el of recEls) {
    if (el.offsetWidth > 0 || el.offsetHeight > 0) {
      return true;
    }
  }

  return false;
}

function updateRecordingUI() {
  const dot = document.getElementById('vd-status-dot');
  const label = document.getElementById('vd-status-text');
  const timer = document.getElementById('vd-live-timer');

  const isRec = checkIsRecording();

  if (isRec) {
    if (!isRecordingActive) {
      isRecordingActive = true;
      recStartTime = Date.now() - (recordingSeconds * 1000);
      if (dot) dot.className = 'vd-rec-dot active';
      if (label) label.innerText = '🔴 Live Recording...';

      if (!timerInterval) {
        timerInterval = setInterval(() => {
          if (isRecordingActive && recStartTime) {
            recordingSeconds = Math.floor((Date.now() - recStartTime) / 1000);
            if (timer) timer.innerText = formatSec(recordingSeconds);
          }
        }, 500);
      }
    }
  } else {
    if (isRecordingActive) {
      isRecordingActive = false;
      if (timerInterval) {
        clearInterval(timerInterval);
        timerInterval = null;
      }
      if (dot) dot.className = 'vd-rec-dot';
      if (label) label.innerText = 'Microphone Ready';
    }
  }
}

setInterval(updateRecordingUI, 200);

document.addEventListener('click', () => {
  setTimeout(updateRecordingUI, 60);
  setTimeout(updateRecordingUI, 200);
  setTimeout(updateRecordingUI, 500);
});

window.resetLocalTimer = function() {
  if (timerInterval) {
    clearInterval(timerInterval);
    timerInterval = null;
  }
  isRecordingActive = false;
  recordingSeconds = 0;
  recStartTime = null;
  const timer = document.getElementById('vd-live-timer');
  const dot = document.getElementById('vd-status-dot');
  const label = document.getElementById('vd-status-text');
  if (timer) timer.innerText = '00:00';
  if (dot) dot.className = 'vd-rec-dot';
  if (label) label.innerText = 'Microphone Ready';
};
</script>
"""

global_matcher = SpeakerMatcher(threshold=0.32)

theme_obj = gr.themes.Default(primary_hue="indigo", neutral_hue="slate")
blocks_kwargs = {"title": "VoiceDiary"}
import inspect
blocks_sig = inspect.signature(gr.Blocks).parameters
if "theme" in blocks_sig:
    blocks_kwargs["theme"] = theme_obj
if "css" in blocks_sig:
    blocks_kwargs["css"] = CSS
if "head" in blocks_sig:
    blocks_kwargs["head"] = HEAD_JS

with gr.Blocks(**blocks_kwargs) as demo:

    transcript_state = gr.State("")
    session_segments_state = gr.State([])
    speech_accumulator = gr.State(np.array([], dtype=np.float32))
    is_speaking_state = gr.State(False)
    silence_counter_state = gr.State(0.0)
    total_timeline_sec = gr.State(0.0)
    total_compute_time = gr.State(0.0)
    ambient_noise_state = gr.State(0.003)
    summary_markdown_state = gr.State("")

    # Header
    gr.HTML(f"""
    <div class="vd-hdr">
      <div class="vd-hdr-left">
        <div class="vd-logo-box">
          <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2.2">
            <path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/>
            <path d="M19 10v2a7 7 0 0 1-14 0v-2"/>
            <line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/>
          </svg>
        </div>
        <div>
          <div class="vd-hdr-title">VoiceDiary</div>
          <div class="vd-hdr-sub">Bilingual Classroom Transcription, Speaker Diarization &amp; Lecture Notes</div>
        </div>
      </div>
      <div class="vd-hw-pill">
        <span class="vd-hw-dot"></span>
        {gpu_name} ({compute_dtype.upper()})
      </div>
    </div>""")

    hidden_rename_payload = gr.Textbox(elem_id="hidden_rename_payload", elem_classes=["vd-hidden-bridge"])
    hidden_rename_trigger = gr.Button("Trigger Rename", elem_id="hidden_rename_trigger", elem_classes=["vd-hidden-bridge"])

    with gr.Row(equal_height=False):
        # Left Sidebar (Speakers & Settings)
        with gr.Column(scale=35, min_width=360):
            sidebar_out = gr.HTML(value="""
            <div class="vd-sec-title first">
              <span>Speakers</span>
              <span class="vd-sec-meta">0 active</span>
            </div>
            <div class="vd-window-box vd-empty-box" style="height: 140px;">
              <div class="vd-empty-title" style="font-size:13px;">No Active Speakers</div>
              <div class="vd-empty-sub">Start speaking to build neural voiceprints</div>
            </div>""")

            gr.HTML("<div class='vd-sec-title'><span>Speech Model &amp; Language</span></div>")
            model_dd = gr.Dropdown(
                choices=list(MODEL_MAP.keys()),
                value='Large-v3-Turbo (809M)',
                label="Whisper Speech Model",
                interactive=True,
                elem_classes=["vd-dropdown"]
            )
            lang_dd = gr.Dropdown(
                choices=list(LANG_MAP.keys()),
                value='Bilingual (Urdu + English)',
                label="Language Output Mode",
                interactive=True,
                elem_classes=["vd-dropdown"]
            )

            # ─── Sidebar Lecture Summary Controls ───
            gr.HTML("<div class='vd-sec-title'><span>AI Lecture Summarizer</span></div>")
            gemini_key_input = gr.Textbox(
                placeholder="Paste Gemini 2.5 API Key (Optional)...",
                value=os.environ.get("GEMINI_API_KEY", ""),
                type="password",
                show_label=False
            )
            generate_summary_btn = gr.Button(
                "✨ Generate Lecture Summary",
                variant="primary",
                size="sm",
                elem_classes=["vd-primary-btn"]
            )

            with gr.Accordion("Advanced Audio & Sensitivity Tuning", open=False):
                max_chunk_sl = gr.Slider(minimum=2.0, maximum=8.0, value=3.5, step=0.5,
                                         label="Max Sentence Length (seconds)",
                                         info="Maximum length before cutting continuous speech")
                pause_ms_sl = gr.Slider(minimum=150, maximum=800, value=280, step=20,
                                        label="Natural Pause Duration (ms)",
                                        info="Natural speech pause threshold to trigger transcription")
                thresh_sl = gr.Slider(minimum=15, maximum=60, value=32, step=1,
                                      label="Speaker Sensitivity (%)",
                                      info="Lower = more speakers | Higher = fewer speakers")


        # Right Main Panel
        with gr.Column(scale=65, min_width=480):
            with gr.Tabs():
                with gr.TabItem("🎙️ Live Classroom Microphone"):
                    gr.HTML("""
                    <div class="vd-rec-bar">
                      <div class="vd-rec-left">
                        <span class="vd-rec-dot" id="vd-status-dot"></span>
                        <span id="vd-status-text">Microphone Ready</span>
                      </div>
                      <span class="vd-rec-timer" id="vd-live-timer">00:00</span>
                    </div>
                    """)
                    audio_mic = gr.Audio(sources=["microphone"], type="numpy", streaming=True,
                                         label="Live Microphone", show_label=False)
                    with gr.Row():
                        clear_live_btn = gr.Button("Clear Session", variant="secondary", size="sm", elem_classes=["vd-secondary-btn"])

                with gr.TabItem("📁 Upload Audio File"):
                    audio_file = gr.Audio(sources=["upload"], type="filepath",
                                          label="Upload Lecture Audio File (.wav, .mp3, .m4a, .flac, .ogg)",
                                          show_label=False)
                    transcribe_file_btn = gr.Button("⚡ Transcribe & Diarize Audio File", variant="primary", elem_classes=["vd-primary-btn"])

            # ─── Classroom Transcript Section (Enclosed Scrollable Box) ───
            transcript_out = gr.HTML(value="""
            <div class="vd-sec-title">
              <span>Classroom Transcript &amp; Lecture Notes</span>
              <span class="vd-sec-meta">0 segments</span>
            </div>
            <div class="vd-window-box vd-empty-box" style="height: 380px;">
              <div class="vd-empty-icon">🎙️</div>
              <div class="vd-empty-title">Live Classroom Transcript Will Stream Here</div>
              <div class="vd-empty-sub">Click 'Record' above or upload an audio file to start transcribing</div>
            </div>""")

            # ─── Lecture Summary Display Section ───
            summary_out = gr.HTML(value="""
            <div class="vd-sec-title">
              <span>Lecture Summary &amp; Key Notes</span>
            </div>
            <div class="vd-window-box vd-empty-box">
              <div class="vd-empty-icon">✨</div>
              <div class="vd-empty-title">No Lecture Summary Generated Yet</div>
              <div class="vd-empty-sub">Click '✨ Generate Lecture Summary' in the sidebar after recording or transcribing</div>
            </div>""")

            # ─── Export Grid (5 Formats) ───
            export_html_out = gr.HTML(value="""
            <div class="vd-sec-title">
              <span>Export &amp; Download Notes</span>
            </div>
            <div class="vd-export-grid">
              <div class="vd-export-card disabled">
                <span class="vd-export-icon">📝</span>
                <div class="vd-export-name">Markdown (.md)</div>
                <div class="vd-export-desc">0 segments</div>
              </div>
              <div class="vd-export-card disabled">
                <span class="vd-export-icon">📄</span>
                <div class="vd-export-name">Plain Text (.txt)</div>
                <div class="vd-export-desc">0 segments</div>
              </div>
              <div class="vd-export-card disabled">
                <span class="vd-export-icon">🎬</span>
                <div class="vd-export-name">Subtitles (.srt)</div>
                <div class="vd-export-desc">0 segments</div>
              </div>
              <div class="vd-export-card disabled">
                <span class="vd-export-icon">📦</span>
                <div class="vd-export-name">JSON (.json)</div>
                <div class="vd-export-desc">0 segments</div>
              </div>
              <div class="vd-export-card disabled">
                <span class="vd-export-icon">✨</span>
                <div class="vd-export-name">Summary (.md)</div>
                <div class="vd-export-desc">Generate in sidebar</div>
              </div>
            </div>""")

    # ─── REAL-TIME LIVE STREAMING HANDLER ───
    def handle_audio_stream(
        stream_data, speech_buf, in_speech, silence_dur, timeline_sec,
        segments_list, compute_time, ambient_noise, summary_md,
        model_choice, lang_choice, max_chunk_s, pause_target_ms, thresh_pct
    ):
        if stream_data is None:
            return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, compute_time, ambient_noise, gr.skip(), gr.skip(), gr.skip(), gr.skip()

        try:
            sr, y = stream_data
            if len(y) == 0:
                return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, compute_time, ambient_noise, gr.skip(), gr.skip(), gr.skip(), gr.skip()

            if y.ndim > 1:
                y = np.mean(y, axis=1)
            if y.dtype != np.float32:
                y = y.astype(np.float32)
                max_val = np.max(np.abs(y)) or 1.0
                if max_val > 1.0:
                    y = y / 32768.0

            y = resample_to_16k(y, sr)
            y = butter_highpass_filter(y)
            chunk_dur = len(y) / 16000.0
            timeline_sec += chunk_dur

            max_speech_samples = int(float(max_chunk_s) * 16000.0) if max_chunk_s else int(3.5 * 16000.0)
            pause_gap_target = float(pause_target_ms) / 1000.0 if pause_target_ms else 0.28
            min_speech_samples = int(16000 * 0.6)

            is_voice, updated_ambient, vad_confidence = is_speech_in_chunk(
                y, silero_vad_model, dev_type=device_type, ambient_rms=ambient_noise, vad_threshold=0.45
            )
            ambient_noise = updated_ambient

            should_cut = False

            if is_voice:
                speech_buf = np.append(speech_buf, y)
                in_speech = True
                silence_dur = 0.0
                if len(speech_buf) >= max_speech_samples:
                    should_cut = True
                    in_speech = False
            else:
                if in_speech:
                    silence_dur += chunk_dur
                    if silence_dur >= pause_gap_target:
                        if len(speech_buf) >= min_speech_samples:
                            should_cut = True
                        in_speech = False
                        silence_dur = 0.0

            if should_cut and len(speech_buf) >= min_speech_samples:
                utterance_audio = speech_buf.copy()
                speech_buf = np.array([], dtype=np.float32)

                t0 = time.time()
                mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
                model = get_model(mkey)
                target_lang, is_urdu_script, is_english_only = LANG_MAP.get(lang_choice, ('ur', False, False))
                is_roman = (lang_choice == 'Roman Urdu (Latin)')
                thresh = float(thresh_pct) * 0.01
                active_prompt = PROMPT_MAP.get(lang_choice, PROMPT_MAP['Bilingual (Urdu + English)'])

                try:
                    segs, info = model.transcribe(
                        utterance_audio,
                        language=target_lang,
                        beam_size=1, best_of=1, temperature=0.0,
                        initial_prompt=active_prompt,
                        condition_on_previous_text=False,
                        without_timestamps=True,
                        no_speech_threshold=0.6,
                        compression_ratio_threshold=2.4,
                        vad_filter=False
                    )

                    valid_words = []
                    for s in segs:
                        raw = s.text.strip()
                        if not raw: continue
                        conf = float(getattr(s, "avg_logprob", getattr(s, "avg_log_prob", 0.0)))
                        no_speech_prob = float(getattr(s, "no_speech_prob", 0.0))
                        if no_speech_prob > 0.65 or conf < -1.2: continue
                        raw = RE_HAL_REPEATS.sub(r'\1', raw).strip()
                        raw = sanitize_script(raw)
                        if not raw or len(raw) < 2: continue
                        valid_words.append(raw)

                    if valid_words:
                        combined_text_list = []
                        for raw in valid_words:
                            if is_urdu_script:
                                txt_clean = UrduNormalizer.normalize(raw)
                            elif is_english_only:
                                txt_clean = post_process_english(raw)
                            elif is_roman:
                                norm_ur = UrduNormalizer.normalize(raw)
                                txt_clean = post_process_english(to_roman_urdu(norm_ur))
                            else:
                                words = raw.split()
                                proc = [UrduNormalizer.normalize(w) if any('\u0600' <= ch <= '\u06FF' for ch in w) else w for w in words]
                                txt_clean = post_process_english(" ".join(proc))
                            if txt_clean and len(txt_clean.strip()) > 0:
                                combined_text_list.append(txt_clean)

                        full_sentence = " ".join(combined_text_list).strip()

                        if full_sentence:
                            slice_dur = len(utterance_audio) / 16000.0
                            emb = extract_speaker_embedding(utterance_audio, embedder, dev_type=device_type)
                            spk_id, spk_name, spk_color, conf = global_matcher.identify(
                                emb, duration=slice_dur, current_time=timeline_sec, threshold_override=thresh
                            )

                            seg_start = max(0.0, timeline_sec - slice_dur)
                            ts = f"{int(seg_start//60):02d}:{int(seg_start%60):02d}"

                            segments_list.append({
                                "speaker": spk_name,
                                "id": spk_id,
                                "color": spk_color,
                                "start": seg_start,
                                "end": timeline_sec,
                                "time": ts,
                                "text": full_sentence
                            })
                except Exception as e:
                    print("Stream transcription error:", e)

                compute_time += (time.time() - t0)
                t_html, s_html, exp_html, plain_txt = render_transcript_ui(
                    segments_list, global_matcher, mkey, compute_time, summary_md=summary_md
                )
                return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, compute_time, ambient_noise, t_html, s_html, exp_html, plain_txt

            return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, compute_time, ambient_noise, gr.skip(), gr.skip(), gr.skip(), gr.skip()

        except Exception as ex:
            print("Stream handling exception:", ex)
            return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, compute_time, ambient_noise, gr.skip(), gr.skip(), gr.skip(), gr.skip()

    # Stream Event Listener with stream_every=1.0 for reliable connection stability
    audio_mic.stream(
        fn=handle_audio_stream,
        inputs=[audio_mic, speech_accumulator, is_speaking_state, silence_counter_state, total_timeline_sec,
                session_segments_state, total_compute_time, ambient_noise_state, summary_markdown_state,
                model_dd, lang_dd, max_chunk_sl, pause_ms_sl, thresh_sl],
        outputs=[speech_accumulator, is_speaking_state, silence_counter_state, total_timeline_sec,
                 session_segments_state, total_compute_time, ambient_noise_state,
                 transcript_out, sidebar_out, export_html_out, transcript_state],
        show_progress="hidden",
        stream_every=1.0
    )

    def reset_live_session():
        global_matcher.reset()
        empty_t, empty_s, empty_e, _ = render_transcript_ui([], global_matcher, 'large-v3-turbo', 0.0)
        empty_sum = render_summary_to_html("")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return (
            None,
            np.array([], dtype=np.float32), False, 0.0, 0.0, [], 0.0, 0.003, "",
            empty_t, empty_s, empty_e, empty_sum, ""
        )

    # Clear Event Listener with smooth instant JS reset callback
    clear_live_btn.click(
        fn=reset_live_session,
        inputs=[],
        outputs=[audio_mic, speech_accumulator, is_speaking_state, silence_counter_state, total_timeline_sec,
                 session_segments_state, total_compute_time, ambient_noise_state, summary_markdown_state,
                 transcript_out, sidebar_out, export_html_out, summary_out, transcript_state],
        show_progress="hidden",
        js="""() => {
            window.resetLocalTimer();
            window._lastRawSummary = '';
            window.spkCustomNames = {};
            const btns = document.querySelectorAll('.audio button, [data-testid="audio"] button, .gradio-audio button');
            for (const b of btns) {
                const txt = (b.innerText || b.getAttribute('aria-label') || '').toLowerCase();
                if (txt.includes('clear') || txt.includes('stop') || txt.includes('delete') || txt.includes('reset')) {
                    try { b.click(); } catch(e) {}
                }
            }
        }"""
    )

    def handle_inline_rename(payload_json, segments_list, compute_time, summary_md, model_choice):
        if payload_json:
            try:
                data = json.loads(payload_json)
                sid = int(data.get("id", 1))
                new_name = str(data.get("name", "")).strip()
                if new_name and sid in global_matcher.speakers:
                    global_matcher.speakers[sid]["name"] = new_name
                    for seg in segments_list:
                        if seg["id"] == sid:
                            seg["speaker"] = new_name
            except Exception as e:
                print("Rename error:", e)

        mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
        t_html, s_html, exp_html, plain_txt = render_transcript_ui(
            segments_list, global_matcher, mkey, compute_time, summary_md=summary_md
        )
        return segments_list, transcript_out, sidebar_out, export_html_out, transcript_state

    hidden_rename_trigger.click(
        fn=handle_inline_rename,
        inputs=[hidden_rename_payload, session_segments_state, total_compute_time, summary_markdown_state, model_dd],
        outputs=[session_segments_state, transcript_out, sidebar_out, export_html_out, transcript_state],
        show_progress="hidden"
    )

    # ─── FILE UPLOAD HANDLER ───
    def handle_file_upload(fpath, model_choice, lang_choice, thresh_pct, vad_ms, summary_md):
        mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
        if not fpath:
            t_html, s_html, exp_html, plain_txt = render_transcript_ui([], global_matcher, mkey, 0.0, summary_md=summary_md)
            return [], t_html, s_html, exp_html, plain_txt

        t0 = time.time()
        data = load_16k_from_file(fpath)
        if len(data) < 4000:
            t_html, s_html, exp_html, plain_txt = render_transcript_ui([], global_matcher, mkey, 0.0, summary_md=summary_md)
            return [], t_html, s_html, exp_html, plain_txt

        model = get_model(mkey)
        target_lang, is_urdu_script, is_english_only = LANG_MAP.get(lang_choice, ('ur', False, False))
        is_roman = (lang_choice == 'Roman Urdu (Latin)')
        thresh = float(thresh_pct) * 0.01
        active_prompt = PROMPT_MAP.get(lang_choice, PROMPT_MAP['Bilingual (Urdu + English)'])

        try:
            segs, info = model.transcribe(
                data,
                language=target_lang,
                beam_size=1, best_of=1, temperature=0.0,
                initial_prompt=active_prompt,
                condition_on_previous_text=False,
                without_timestamps=False,
                no_speech_threshold=0.6,
                compression_ratio_threshold=2.4,
                vad_filter=True,
                vad_parameters=dict(min_silence_duration_ms=int(vad_ms) if vad_ms else 280)
            )

            file_matcher = SpeakerMatcher(threshold=thresh)
            file_segments = []

            for s in segs:
                raw = s.text.strip()
                if not raw: continue
                conf = float(getattr(s, "avg_logprob", getattr(s, "avg_log_prob", 0.0)))
                no_speech_prob = float(getattr(s, "no_speech_prob", 0.0))
                if no_speech_prob > 0.65 or conf < -1.2: continue
                raw = RE_HAL_REPEATS.sub(r'\1', raw).strip()
                raw = sanitize_script(raw)
                if not raw or len(raw) < 2: continue

                if is_urdu_script:
                    txt_clean = UrduNormalizer.normalize(raw)
                elif is_english_only:
                    txt_clean = post_process_english(raw)
                elif is_roman:
                    norm_ur = UrduNormalizer.normalize(raw)
                    txt_clean = post_process_english(to_roman_urdu(norm_ur))
                else:
                    words = raw.split()
                    proc = [UrduNormalizer.normalize(w) if any('\u0600' <= ch <= '\u06FF' for ch in w) else w for w in words]
                    txt_clean = post_process_english(" ".join(proc))

                if not txt_clean or len(txt_clean.strip()) == 0:
                    continue

                start_idx = max(0, int(s.start * 16000))
                end_idx = min(len(data), int(s.end * 16000))
                audio_slice = data[start_idx:end_idx]

                if len(audio_slice) >= 4000:
                    emb = extract_speaker_embedding(audio_slice, embedder, dev_type=device_type)
                    spk_id, spk_name, spk_color, spk_conf = file_matcher.identify(
                        emb, duration=(s.end - s.start), current_time=s.end, threshold_override=thresh
                    )
                else:
                    spk_id = 1
                    spk_name = "Speaker 1"
                    spk_color = "#6366F1"

                ts = f"{int(s.start//60):02d}:{int(s.start%60):02d}"
                file_segments.append({
                    "speaker": spk_name,
                    "id": spk_id,
                    "color": spk_color,
                    "start": s.start,
                    "end": s.end,
                    "time": ts,
                    "text": txt_clean
                })

            total_t = time.time() - t0
            t_html, s_html, exp_html, plain_txt = render_transcript_ui(
                file_segments, file_matcher, mkey, total_t, summary_md=summary_md
            )
            return file_segments, t_html, s_html, exp_html, plain_txt
        except Exception as e:
            print("File transcribe error:", e)
            t_html, s_html, exp_html, plain_txt = render_transcript_ui([], global_matcher, mkey, 0.0, summary_md=summary_md)
            return [], t_html, s_html, exp_html, plain_txt

    transcribe_file_btn.click(
        fn=handle_file_upload,
        inputs=[audio_file, model_dd, lang_dd, thresh_sl, pause_ms_sl, summary_markdown_state],
        outputs=[session_segments_state, transcript_out, sidebar_out, export_html_out, transcript_state],
        show_progress="hidden"
    )

    # ─── LECTURE SUMMARY GENERATION HANDLER ───
    def generate_ai_lecture_summary(segments_list, api_key_input, model_choice, compute_time):
        if not segments_list:
            msg = """<div class="vd-sec-title">
              <span>Lecture Summary &amp; Key Notes</span>
            </div>
            <div class="vd-window-box vd-empty-box" style="height: 220px; border-color: rgba(239,68,68,0.3);">
              <div class="vd-empty-icon">⚠️</div>
              <div class="vd-empty-title" style="color:#F87171;">No Classroom Transcript Available</div>
              <div class="vd-empty-sub">Please record live classroom audio or upload an audio file before generating summary notes.</div>
            </div>"""
            yield "", msg, gr.skip()
            return

        # 1. Immediately display the smooth AI loading spinner card without disappearing
        yield "", render_summary_loading_html(), gr.skip()

        # 2. Synthesize lecture notes with Gemini
        title = "University Classroom Lecture"
        summarizer = GeminiSummarizer(api_key=api_key_input)
        res = summarizer.summarize_lecture(segments_list, lecture_title=title)

        # 3. Render final output
        if res.get("success"):
            raw_md = res.get("summary_markdown", "")
            summary_html = render_summary_to_html(raw_md)
            mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
            _, _, exp_html, _ = render_transcript_ui(
                segments_list, global_matcher, mkey, compute_time, summary_md=raw_md
            )
            yield raw_md, summary_html, exp_html
        else:
            err = res.get("message", "Unknown error generating summary")
            err_html = f"""<div class="vd-sec-title">
              <span>Lecture Summary &amp; Key Notes</span>
            </div>
            <div class="vd-window-box vd-empty-box" style="height: 220px; border-color: rgba(239,68,68,0.3);">
              <div class="vd-empty-icon">❌</div>
              <div class="vd-empty-title" style="color:#F87171;">Summary Generation Failed</div>
              <div class="vd-empty-sub" style="color:#FDA4AF; max-width:460px;">{err}</div>
            </div>"""
            yield "", err_html, gr.skip()

    generate_summary_btn.click(
        fn=generate_ai_lecture_summary,
        inputs=[session_segments_state, gemini_key_input, model_dd, total_compute_time],
        outputs=[summary_markdown_state, summary_out, export_html_out],
        show_progress="hidden"
    )

launch_kwargs = {
    "share": True,
    "debug": False,
    "show_error": True,
}
launch_sig = inspect.signature(demo.launch).parameters
if "theme" in launch_sig and "theme" not in blocks_kwargs:
    launch_kwargs["theme"] = theme_obj
if "css" in launch_sig and "css" not in blocks_kwargs:
    launch_kwargs["css"] = CSS
if "head" in launch_sig and "head" not in blocks_kwargs:
    launch_kwargs["head"] = HEAD_JS

demo.queue(default_concurrency_limit=1, max_size=100).launch(**launch_kwargs)
